# Data Ingestion and Preprocessing: pii-masking-300k

This notebook focuses on setup for my project with data ingestion and preprocessing. The two datasets I am using are `ai4privacy/pii-masking-300k`. I want it to be in a format with a source_text field, a tagged_text field with HTML-like tags, and I want to cover a variety of useful cases for training and testing.

## Setup

In [2]:
## Imports
import re
import json
import torch
from collections import Counter
from datasets import load_dataset, Dataset

## Load the dataset

Loading the dataset from HuggingFace.

In [3]:

ds = load_dataset("ai4privacy/pii-masking-300k")

README.md: 0.00B [00:00, ?B/s]

data/train/1english_openpii_30k.jsonl:   0%|          | 0.00/103M [00:00<?, ?B/s]

data/train/dutch_openpii_28k.jsonl:   0%|          | 0.00/102M [00:00<?, ?B/s]

data/train/french_openpii_31k.jsonl:   0%|          | 0.00/114M [00:00<?, ?B/s]

data/train/german_openpii_30k.jsonl:   0%|          | 0.00/108M [00:00<?, ?B/s]

...

data/validation/german_openpii_8k.jsonl:   0%|          | 0.00/29.2M [00:00<?, ?B/s]

data/validation/italian_openpiii_8k.json(…):   0%|          | 0.00/28.3M [00:00<?, ?B/s]

data/validation/spanish_openpii_8k.jsonl:   0%|          | 0.00/27.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/177677 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/47728 [00:00<?, ? examples/s]

In [4]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set'],
        num_rows: 177677
    })
    validation: Dataset({
        features: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set'],
        num_rows: 47728
    })
})


## Exploratory: Checking the current contents and structure of the data

I wanted to see what data is already captured before figuring out how it needs to be transformed. This includes checking the tags, languages, and features. I ultimately decided to limit this dataset to English since the NP tests are administered in English.

In [6]:
def export_pii_masking(ds):
    """Write pii-masking-300k dataset from hugging face as txt files"""
    dataset = ds
    for i, row in enumerate(dataset):
        with open(f"{str(i)}.txt", "w", encoding="utf-8") as out_file:
            out_file.write(row["source_text"])

In [7]:
# Load
dataset = load_dataset("ai4privacy/pii-masking-300k", split="train")

In [8]:
# Check available fields first
print(dataset.features)

{'source_text': Value('string'), 'target_text': Value('string'), 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}), 'span_labels': Value('string'), 'mbert_text_tokens': List(Value('string')), 'mbert_bio_labels': List(Value('string')), 'id': Value('string'), 'language': Value('string'), 'set': Value('string')}


In [9]:
# Count languages
lang_counts = Counter(dataset["language"])  # adjust field name if needed
for lang, count in sorted(lang_counts.items(), key=lambda x: -x[1]):
    print(f"{lang:20s}: {count:6d}")

French              :  31447
German              :  29976
English             :  29908
Italian             :  29066
Spanish             :  28847
Dutch               :  28433


In [10]:
## Filtering for English-only -- narrowing the scope for this assignment
english_dataset = dataset.filter(lambda x: x["language"] == "English")
print(f"English examples: {len(english_dataset)}")

Filter:   0%|          | 0/177677 [00:00<?, ? examples/s]

English examples: 29908


In [12]:
print(english_dataset)

Dataset({
    features: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set'],
    num_rows: 29908
})


In [13]:
# Count all labels of PII in the dataset
privacy_mask_all = dataset["privacy_mask"]
unique_labels = set()
for pii_list in privacy_mask_all:
    for item in pii_list:
        label = item.get('label', None)
        if label is not None:
            unique_labels.add(label) 

In [10]:
print(unique_labels)

{'CITY', 'SECADDRESS', 'DRIVERLICENSE', 'PASS', 'IP', 'GIVENNAME2', 'SEX', 'SOCIALNUMBER', 'STREET', 'LASTNAME2', 'LASTNAME3', 'GEOCOORD', 'STATE', 'LASTNAME1', 'DATE', 'BUILDING', 'IDCARD', 'POSTCODE', 'TIME', 'USERNAME', 'TITLE', 'CARDISSUER', 'EMAIL', 'GIVENNAME1', 'TEL', 'BOD', 'PASSPORT', 'COUNTRY'}


I filtered for an English-only dataset and got all of the tags that were found in that dataset. There are many more tags than I am planning to use, so I next created a mapping to the tags I want to include in my tagging system.

In [14]:
## Mapping
TAG_MAPPING = {
    # NAME
    "GIVENNAME1": "NAME",
    "GIVENNAME2": "NAME",
    "LASTNAME1": "NAME",
    "LASTNAME2": "NAME",
    "LASTNAME3": "NAME",
    "TITLE": "TITLE",
    # DATE
    "BOD": "DOB",
    "DATE": "DATE",
    "TIME": "TIME",
    # CONTACT
    "EMAIL": "EMAIL",
    "TEL": "PHONE",
    # ADDRESS
    "STREET": "ADDRESS",
    "BUILDING": "ADDRESS",
    "SECADDRESS": "ADDRESS",
    "POSTCODE": "ADDRESS",
    # LOCATION
    "CITY": "LOCATION",
    "STATE": "LOCATION",
    "COUNTRY": "LOCATION",
    "GEOCOORD": "LOCATION",
    # IDs
    "SOCIALNUMBER": "SSN",
    "IDCARD": "UNIQUE_ID",
    "PASSPORT": "UNIQUE_ID",
    "DRIVERLICENSE": "LICENSE_NUMBER",
    "IP": "IP_ADDRESS",

    # EXTRA
    "USERNAME": "USERNAME",
    "PASS": "PASS",
    "SEX": "SEX",
    "CARDISSUER": "CARDISSUER",
}

In [35]:
def reconstruct_inline_tags(example, tag_mapping):
    """
    Reconstruct inline XML tags using character-level span indices.
    Processes spans right to left to preserve index validity.
    """
    source_text  = example["source_text"]
    privacy_mask = example["privacy_mask"]

    # Sort by start index descending (right to left)
    sorted_mask = sorted(privacy_mask, key=lambda x: x["start"], reverse=True)

    text = source_text
    pii_tags = set()
    pii_present = False
    pii_dummies = {
        'NAME': False,
        'TITLE': False,
        'DOB': False,
        'DATE': False,
        'TIME': False,
        'EMAIL': False,
        'PHONE': False,
        'ADDRESS': False,
        'LOCATION': False,
        'SSN': False,
        'UNIQUE_ID': False,
        'LICENSE_NUMBER': False,
        'IP_ADDRESS': False,
        'USERNAME': False,
        'PASS': False,
        'SEX': False,
        'CARDISSUER': False     
    }

    for entity in sorted_mask:
        start = entity["start"]
        end   = entity["end"]
        value = entity["value"]
        label = entity["label"]

        # Confirm span matches value, otherwise skip entire entry because incorrectly tagged
        span_text = text[start:end]
        if span_text != value:
            return None

        mapped_tag = tag_mapping.get(label, None)

        ## Skip if not included in my tag schema
        if mapped_tag is None:
            continue
        else:
            # Replace span with inline tagged version
            text = text[:start] + f"<{mapped_tag}>{value}</{mapped_tag}>" + text[end:]
            pii_tags.add(mapped_tag)
            pii_dummies[mapped_tag] = True
            
    ## Checking if the set of pii has elements
    if pii_tags:
        pii_present = True
    data = {
        "source_text": source_text,
        "contains_pii": pii_present,
        "pii_tags": list(pii_tags),
        "ai4privacy_text": example["target_text"],
        "tagged_text": text,
    }
    return data | pii_dummies

Before applying to the whole dataset, I tested one example.

In [36]:
## Test:
example_text1 = english_dataset[0]
test1 = reconstruct_inline_tags(example_text1, TAG_MAPPING)

In [37]:
test1['NAME']

False

Since that example was successful, I applied the tag mapping to the entire dataset. I then exported this dataset for future use. If the text included in my schema, I skipped it and excluded it from the dataset.

In [38]:
print("\nProcessing examples...")
processed = []
skipped   = 0

for example in english_dataset:
    result = reconstruct_inline_tags(example, TAG_MAPPING)
    if result is None:
        skipped += 1
        continue
    processed.append(result)

hf_dataset = Dataset.from_list(processed)
hf_dataset.save_to_disk("../data/ai4privacy/pii-masking-300k_processed")
print("\nSaved to ../data/ai4privacy/pii-masking-300k_processed")

print(f"Kept:    {len(processed)}")
print(f"Skipped: {skipped}")


Processing examples...


Saving the dataset (0/1 shards):   0%|          | 0/29713 [00:00<?, ? examples/s]


Saved to ../data/ai4privacy/pii-masking-300k_processed
Kept:    29713
Skipped: 195


Here is an example of one entry from the dataset.

In [72]:
processed[50]

{'source_text': 'Inventory Records - Equipment and Supplies\n\nDate of Inventory: 05:48\n\nID: CB22356OB\n- Email: SSJEB@yahoo.com\n- Country: GB\n- Building Number: 499\n- Street: Bonby Lane\n- City: York\n- State: ENG\n- Postcode: YO19\n\nID: GM26061IE\n- Email: ZB@yahoo.com\n- Country: GB\n- Building Number: 80\n- Street: Blenheim Palace\n- City: Woodstock\n- State: ENG\n- Postcode: OX20\n- Secondary Address: RV 416\n\nID: NV22510SC\n- Email: 30viria.alatas@gmail.com\n- Country: United Kingdom\n- Building Nu',
 'contains_pii': True,
 'pii_tags': ['TIME', 'EMAIL', 'UNIQUE_ID', 'ADDRESS', 'LOCATION'],
 'ai4privacy_text': 'Inventory Records - Equipment and Supplies\n\nDate of Inventory: [TIME]\n\nID: [IDCARD]\n- Email: [EMAIL]\n- Country: [COUNTRY]\n- Building Number: [BUILDING]\n- Street: [STREET]\n- City: [CITY]\n- State: [STATE]\n- Postcode: [POSTCODE]\n\nID: [IDCARD]\n- Email: [EMAIL]\n- Country: [COUNTRY]\n- Building Number: [BUILDING]\n- Street: [STREET]\n- City: [CITY]\n- State:

This is the basis for the dataset I used during training. At the time of training, I made sure to balance the datset, which you can see in my training file.